# English lexical baselines

Document-level multilabel narrative classification with word TF-IDF and one-vs-rest logistic regression.

The experiment compares a constant `Other` predictor with two decoding rules applied to the same fitted classifiers: a fixed probability threshold of 0.5 and separate coarse/fine thresholds selected on inner validation. The primary metric is mean per-document fine-label F1 under the 96-label project taxonomy.

**Execution:** attach `training_NLP.zip`, `test_NLP.zip` and `kit_setup_nlp.zip` (archived or extracted), then run sections 1–12 sequentially on CPU. This notebook reconstructs its own setup; it does not rely on another notebook's memory. Dependencies include NumPy, pandas, scikit-learn, joblib and PyTorch for the shared seed helper. No packages are installed or pretrained models downloaded automatically.

**Outputs:** a unique experiment directory containing configuration, models, validation searches, probabilities, out-of-fold predictions, metrics and a ZIP archive. Running section 10 again starts a new experiment directory.

Test texts are read only for overlap screening; their annotations are not loaded. The supplied test set was consulted during earlier development and is not an untouched benchmark. The extended fine-label taxonomy is a project definition.

**Reference environment:** the archived experiment used Python 3.13.18, NumPy 2.1.3, pandas 2.3.3 and scikit-learn 1.6.1 on Linux. A fixed seed alone does not ensure identical refitted predictions under different versions or platforms. Preserve archived predictions when reproducing reported tables; record a new environment for each refit.

## 1. Input and output paths

Defaults target Kaggle. For local execution, set `NLP_INPUT_ROOT` and `NLP_OUTPUT_ROOT` before running this cell. Optional source overrides are `NLP_TRAIN_SOURCE`, `NLP_TEST_SOURCE` and `NLP_KIT_DIR`.

An extracted training/test source override must point to the directory containing `training/` or `test/`, respectively. A kit override must point directly to the directory containing `nlp_setup.py`. If multiple copies are attached, choose explicit paths.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil

INPUT_ROOT = Path(os.environ.get('NLP_INPUT_ROOT', '/kaggle/input'))
OUTPUT_ROOT = Path(os.environ.get('NLP_OUTPUT_ROOT', '/kaggle/working/nlp_setup'))
TRAIN_OVERRIDE = os.environ.get('NLP_TRAIN_SOURCE') or None
TEST_OVERRIDE = os.environ.get('NLP_TEST_SOURCE') or None
KIT_OVERRIDE = os.environ.get('NLP_KIT_DIR') or None
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_ROOT)
print('Output:', OUTPUT_ROOT)


## 2. Load the support kit and validate the taxonomy

An extracted kit or a ZIP archive is accepted. An archived kit is extracted into the output directory with path-containment checks. A unique support-module directory is required, including when the archive contains a wrapper directory. The selected module is loaded explicitly to avoid reusing a module from an earlier session import.

In [ ]:
if KIT_OVERRIDE:
    KIT_DIR = Path(KIT_OVERRIDE)
else:
    found = sorted({p.parent for p in INPUT_ROOT.rglob('nlp_setup.py')})
    if len(found) == 1:
        KIT_DIR = found[0]
    elif len(found) > 1:
        raise ValueError('Più kit trovati: impostare KIT_OVERRIDE nella cella 1.')
    else:
        archives = list(INPUT_ROOT.rglob('kit_setup_nlp.zip'))
        if len(archives) != 1:
            raise ValueError('Attach exactly one kit_setup_nlp.zip or set KIT_OVERRIDE.')
        KIT_DIR = OUTPUT_ROOT / '_kit'
        KIT_DIR.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as z:
            for info in z.infolist():
                target = (KIT_DIR / info.filename).resolve()
                if not target.is_relative_to(KIT_DIR.resolve()):
                    raise ValueError('Invalid path in kit archive.')
            z.extractall(KIT_DIR)
        extracted = sorted({p.parent for p in KIT_DIR.rglob('nlp_setup.py')})
        if len(extracted) != 1:
            raise ValueError('Extracted archive must contain exactly one support kit.')
        KIT_DIR = extracted[0]

import importlib.util
if not (KIT_DIR / 'nlp_setup.py').is_file():
    raise FileNotFoundError('KIT_DIR must contain nlp_setup.py.')
spec = importlib.util.spec_from_file_location('nlp_setup', KIT_DIR / 'nlp_setup.py')
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
import numpy as np
import pandas as pd
from IPython.display import display

taxonomy = json.loads((KIT_DIR / 'tassonomia_progetto_v1.json').read_text(encoding='utf-8'))
setup.require(len(taxonomy['coarse']) == 22 and len(taxonomy['fine']) == 96, 'Unexpected taxonomy.')
setup.require(len(set(taxonomy['coarse'])) == 22, 'Duplicate coarse labels.')
setup.require(len(set(taxonomy['fine'])) == 96, 'Duplicate fine labels.')
setup.require(all(l == 'Other' or l.rsplit(': ', 1)[0] in taxonomy['coarse'] for l in taxonomy['fine']),
              'Missing parent label in taxonomy.')
environment = setup.environment_info()
setup.save_json(OUTPUT_ROOT / 'environment.json', environment)
print('Kit:', KIT_DIR)
print('Classes: 22 coarse / 96 fine. Environment:')
display(environment)


## 3. Load training annotations and test texts

Training labels are validated against the project taxonomy. Repeated labels within a document are deduplicated while retaining a fingerprint of the original annotation row. Unknown labels raise an error.

Test annotations are not read. Training records and test texts must match the previously frozen data fingerprints.

In [ ]:
training_source = setup.locate_source(INPUT_ROOT, 'training_NLP.zip', 'training', TRAIN_OVERRIDE)
test_source = setup.locate_source(INPUT_ROOT, 'test_NLP.zip', 'test', TEST_OVERRIDE)
try:
    train_records = setup.load_training(training_source, taxonomy)
    course_texts = setup.load_course_texts(test_source)
finally:
    for source in (training_source, test_source):
        if source.archive is not None:
            source.archive.close()
expected = json.loads((KIT_DIR / 'expected_data_fingerprints.json').read_text(encoding='utf-8'))
setup.require(setup.fingerprint_records(train_records) == expected['training_records'],
              'Training data do not match the verified version.')
setup.require(setup.fingerprint_records(course_texts) == expected['course_texts'],
              'Test texts do not match the verified version.')

data_counts = pd.DataFrame([
    {'language': lang,
     'training': sum(r['language'] == lang for r in train_records),
     'test_texts_only': sum(r['language'] == lang for r in course_texts)}
    for lang in setup.LANGUAGES
])
display(data_counts)
repeated_docs = sum(r['repeated_labels_removed'] > 0 for r in train_records)
print('Training documents with repeated labels deduplicated:', repeated_docs)


## 4. Duplicate and near-duplicate screening

The lexical audit combines whitespace-normalized exact matches with within-language word-trigram TF-IDF cosine similarity of at least 0.80. Candidate links form connected groups. Training documents connected to supplied test texts are excluded from auxiliary training.

The audit threshold is fixed independently of classifier predictions. Audit TF-IDF features are never reused as classifier inputs. This screening cannot rule out translations or all paraphrases, and candidate similarity is not proof of semantic equivalence.

In [ ]:
similarity = setup.audit_similarity(train_records, course_texts, threshold=0.80)
reference = json.loads((KIT_DIR / 'near_duplicate_audit_reference.json').read_text(encoding='utf-8'))
pair_keys = lambda a: {(p['a'], p['b']) for p in a['pairs']}
setup.require(pair_keys(similarity) == pair_keys(reference),
              'Near-duplicate candidates differ from the reference: check inputs and versions.')
setup.require(similarity['excluded_training_keys'] == reference['excluded_training_keys'],
              'Exclusions differ from the frozen reference.')
setup.save_json(OUTPUT_ROOT / 'near_duplicate_audit.json', similarity)
pd.DataFrame(similarity['pairs']).to_csv(OUTPUT_ROOT / 'similarity_pairs.csv', index=False)
print('Candidate pairs:', len(similarity['pairs']))
print('Excluded auxiliary documents:', len(similarity['excluded_training_keys']))
display(pd.DataFrame({'excluded_document': similarity['excluded_training_keys']}))


## 5. Validate the fixed grouped splits

The version-2 manifest assigns the 399 English articles to 395 groups. In particular, `EN_CC_200107.txt` and `EN_CC_200093.txt`, which differ only in quotation marks, share a group.

Each English article appears in exactly one outer-evaluation partition across the three folds. Within a fold, detected groups must not cross training, validation and outer-evaluation partitions. No documents are removed to balance label frequencies.

In [ ]:
manifest = setup.validate_manifest(train_records, similarity, KIT_DIR / 'split_manifest_v2.csv')
split_counts = manifest.groupby(['fold', 'partition']).size().unstack()
display(split_counts[['inner_train', 'inner_validation', 'outer_evaluation']])
manifest.to_csv(OUTPUT_ROOT / 'verified_split_manifest_v2.csv', index=False)
print('OK: 399 English documents; each assigned to outer evaluation exactly once across three folds.')
print('OK: exact-match and candidate English groups do not cross partitions within a fold.')


## 6. Prepare one fold without training

This example builds the first English-only training partition and its multilabel targets. Inner validation is reserved for model and threshold selection; outer predictions are generated only after those choices are fixed. The partitioning code does not technically prevent access to outer labels, so this separation is a methodological requirement.

A fixed seed improves reproducibility but does not guarantee identical results across software versions and hardware.

In [ ]:
FOLD = 1
TRAIN_SEED = 42
MULTILINGUAL = False
setup.seed_everything(TRAIN_SEED)
fold_data = setup.make_fold(train_records, manifest, similarity, FOLD, MULTILINGUAL)
y_train_coarse = setup.binary_labels([r['coarse'] for r in fold_data['inner_train']], taxonomy['coarse'])
y_train_fine = setup.binary_labels([r['fine'] for r in fold_data['inner_train']], taxonomy['fine'])
print('Fold:', FOLD, '| Multilingual:', MULTILINGUAL)
print({role: len(rows) for role, rows in fold_data.items()})
print('Training targets:', y_train_coarse.shape, y_train_fine.shape)
print('Il seed migliora la riproducibilità, ma non garantisce identità tra hardware e versioni diverse.')


## 7. Verify metric compatibility on synthetic cases

The project metric is checked against the original v3 scorer on shared labels and valid synthetic inputs. The comparison covers mean per-document F1 and its document-level standard deviation; the latter is not a confidence interval.

The original scorer ignores labels outside its vocabulary, whereas the strict project implementation rejects unknown labels. Agreement on shared labels does not make the 96-label project taxonomy equivalent to the official taxonomy.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('official_st2_scorer', KIT_DIR / 'official_v3/subtask2_scorer.py')
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
rng = np.random.default_rng(20261103)
for level, filename in [('coarse', 'subtask2_narratives.txt'), ('fine', 'subtask2_subnarratives.txt')]:
    classes = (KIT_DIR / 'official_v3' / filename).read_text(encoding='utf-8').splitlines()
    gold = [[], [classes[0]], [classes[0]], [classes[0], classes[1]]]
    pred = [[], [classes[0]], [], [classes[1]]]
    for _ in range(30):
        gold.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
        pred.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
    ours = setup.sample_f1_strict(gold, pred, classes)
    theirs = official.evaluate(gold, pred, classes)
    setup.require(np.allclose([ours['mean'], ours['document_std']], theirs, atol=1e-12),
                  'Scorers disagree on synthetic cases.')
print('OK: metric agrees with v3 on synthetic cases using shared labels.')
try:
    setup.sample_f1_strict([['UNKNOWN_LABEL']], [[]], taxonomy['fine'])
except ValueError:
    print('OK: unknown labels are rejected.')
else:
    raise AssertionError('Unknown label was not rejected.')


## 8. Export the setup summary

`setup_summary.json` records counts, fold sizes, environment and limitations. Preserve this file together with the verified manifest and audit outputs. Subsequent notebooks must reconstruct their required variables in their own sessions; saving this notebook does not transfer its in-memory variables.

No classifier training is performed here. The support kit provides `create_run_directory` for storing later experiments in separate directories.

In [ ]:
fold_sizes = {}
for fold in (1, 2, 3):
    fold_sizes[str(fold)] = {}
    for multi in (False, True):
        parts = setup.make_fold(train_records, manifest, similarity, fold, multi)
        fold_sizes[str(fold)]['multilingual' if multi else 'EN_only'] = {k:len(v) for k,v in parts.items()}
summary = {
    'status': 'SETUP_OK', 'split_version': 'v2', 'taxonomy_version': 'progetto_v1_96fine',
    'training_documents_original': len(train_records),
    'course_test_texts': len(course_texts), 'course_test_labels_loaded': False,
    'coarse_classes': len(taxonomy['coarse']), 'fine_classes': len(taxonomy['fine']),
    'similarity_candidate_pairs': len(similarity['pairs']),
    'auxiliary_documents_excluded': len(similarity['excluded_training_keys']),
    'documents_with_repeated_labels': repeated_docs,
    'fold_sizes': fold_sizes, 'environment': environment,
    'limitations': ['Lexical screen does not rule out translations or all paraphrases.',
                    'The supplied test set was consulted during earlier development; it is not an untouched benchmark.',
                    'Extended taxonomy is a project definition, not an official new release.'],
    'training_started': False
}
setup.save_json(OUTPUT_ROOT / 'setup_summary.json', summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Setup complete. Summary:', OUTPUT_ROOT / 'setup_summary.json')


## 9. Baseline configuration and functions

Word unigrams and bigrams are represented using TF-IDF with sublinear term frequency and at most 15,000 features. Independent L2-regularized logistic regressions are fitted for each label.

A label absent from inner training receives constant probability zero; a label positive in every inner-training document receives probability one. All labels remain in the evaluation vocabulary.

Both TF-IDF variants select labels with probability strictly above the threshold. If none qualifies, the highest-scoring label is returned. No hierarchy correction or `Other` exclusivity rule is applied. Ties in validation F1 select the larger threshold. This cell defines functions without training classifiers.

In [ ]:
import time
import warnings
import hashlib
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score, precision_score, recall_score

BASELINE_CONFIG = {
    'system': 'baseline_EN_v1', 'fold': 'all3', 'seed': 42,
    'split_version': 'v2', 'taxonomy_version': 'project_v1_96fine',
    'multilingual': False,
    'vectorizer': {'max_features': 15000, 'ngram_range': (1, 2), 'sublinear_tf': True},
    'logistic': {'C': 1.0, 'solver': 'liblinear', 'max_iter': 2000, 'random_state': 42},
    'threshold_grid': [.05, .08, .10, .12, .15, .18, .20, .25, .30, .35, .40, .45, .50],
    'threshold_comparison': 'strictly greater',
    'fallback': 'argmax if no label exceeds threshold, for BOTH TF-IDF systems',
    'threshold_tie_break': 'larger threshold',
    'selection': 'sample-F1 on inner validation, separately for coarse and fine',
    'postprocessing': 'none beyond argmax fallback; Other may coexist with specific labels',
    'constant_baseline': 'Other at both levels',
    'unobserved_labels': 'constant probability 0; labels present in every training row -> 1',
}

def fit_binary_models(X, y):
    models = []
    for j in range(y.shape[1]):
        unique = np.unique(y[:, j])
        if len(unique) == 1:
            models.append(float(unique[0]))
        else:
            model = LogisticRegression(**BASELINE_CONFIG['logistic'])
            with warnings.catch_warnings():
                warnings.simplefilter('error', ConvergenceWarning)
                model.fit(X, y[:, j])
            models.append(model)
    return models

def predict_probabilities(models, X):
    probs = np.column_stack([np.full(X.shape[0], model) if isinstance(model, float)
                             else model.predict_proba(X)[:, list(model.classes_).index(1)]
                             for model in models])
    setup.require(np.isfinite(probs).all() and ((probs >= 0) & (probs <= 1)).all(),
                  'Probabilità non valide.')
    return probs

def decode_probabilities(probs, threshold, classes):
    setup.require(probs.shape[1] == len(classes), 'Invalid class order or dimensions.')
    binary = (probs > threshold).astype(int)
    empty = np.where(binary.sum(axis=1) == 0)[0]
    binary[empty, np.argmax(probs[empty], axis=1)] = 1
    return [[classes[j] for j in np.flatnonzero(row)] for row in binary]

def choose_threshold(gold, probs, classes):
    search = []
    for threshold in BASELINE_CONFIG['threshold_grid']:
        predicted = decode_probabilities(probs, threshold, classes)
        score = setup.sample_f1_strict(gold, predicted, classes)['mean']
        search.append({'threshold': threshold, 'validation_sample_f1': score})
    best = max(search, key=lambda r: (r['validation_sample_f1'], r['threshold']))
    return best['threshold'], search

def write_predictions(path, records, coarse, fine):
    setup.require(len(records) == len(coarse) == len(fine), 'Incomplete predictions.')
    with open(path, 'w', encoding='utf-8', newline='') as f:
        for row, c, s in zip(records, coarse, fine):
            f.write(row['document_id']+'\t'+';'.join(c)+'\t'+';'.join(s)+'\n')

def evaluation_metrics(records, predicted, official_classes):
    result = {}
    for level in ('coarse', 'fine'):
        gold = [r[level] for r in records]
        pred = predicted[level]
        classes = taxonomy[level]
        y = setup.binary_labels(gold, classes)
        p = setup.binary_labels(pred, classes)
        result['F1_'+level] = setup.sample_f1_strict(gold, pred, classes)['mean']
        result['mean_predicted_'+level] = float(p.sum(axis=1).mean())
        allowed = set(official_classes[level])
        gold_v3 = [[l for l in ls if l in allowed] for ls in gold]
        pred_v3 = [[l for l in ls if l in allowed] for ls in pred]
        result['F1_'+level+'_v3'] = float(official.evaluate(gold, pred, official_classes[level])[0])
        # Check explicit projection agrees with the original scorer's silent ignore behavior.
        assert np.isclose(result['F1_'+level+'_v3'], setup.sample_f1_strict(gold_v3, pred_v3, official_classes[level])['mean'])
        result['gold_labels_ignored_v3_'+level] = sum(len(set(ls)-allowed) for ls in gold)
        result['pred_labels_ignored_v3_'+level] = sum(len(set(ls)-allowed) for ls in pred)
        if level == 'fine':
            supported = y.sum(axis=0) > 0
            result['micro_F1_fine'] = float(f1_score(y, p, average='micro', zero_division=0))
            result['macro_F1_fine_gold_supported'] = float(f1_score(y[:,supported],p[:,supported],average='macro',zero_division=0))
            result['macro_fine_class_count'] = int(supported.sum())
            result['micro_precision_fine'] = float(precision_score(y,p,average='micro',zero_division=0))
            result['micro_recall_fine'] = float(recall_score(y,p,average='micro',zero_division=0))
    fine_specific = sum(l != 'Other' for ls in predicted['fine'] for l in ls)
    orphan = sum(l != 'Other' and l.rsplit(': ',1)[0] not in coarse
                 for coarse, fine in zip(predicted['coarse'],predicted['fine']) for l in fine)
    result['fine_without_parent_rate'] = orphan / fine_specific if fine_specific else 0.0
    for level in ('coarse','fine'):
        result['mixed_Other_rate_'+level] = float(np.mean(['Other' in ls and len(ls)>1 for ls in predicted[level]]))
    return result

# Known edge cases: threshold equality, argmax fallback, unknown label rejection.
assert decode_probabilities(np.array([[.5,.4],[.1,.9]]),.5,['A','B']) == [['A'],['B']]
assert np.isclose(setup.sample_f1_strict([['A','B']],[['A']],['A','B'])['mean'],2/3)
print('BASELINE_CONFIG_OK — 3 sistemi, training EN, 3 fold, seed 42.')
print('Threshold selection uses inner validation only. No training started in this cell.')


## 10. Fit three folds and select validation thresholds

Vocabulary, IDF weights and classifiers are fitted exclusively on `inner_train`. Thresholds are selected separately for coarse and fine labels using only `inner_validation`. No refitting on the combined training and validation sets is performed.

Each fold stores its models, class order, training/validation identifiers, threshold search and validation probabilities. Convergence warnings terminate the run. Outer labels are not used for fitting or threshold selection.

In [ ]:
RUN_DIR = setup.create_run_directory(OUTPUT_ROOT, BASELINE_CONFIG)
setup.save_json(RUN_DIR / 'taxonomy.json', taxonomy)
shutil.copyfile(KIT_DIR / 'split_manifest_v2.csv', RUN_DIR / 'split_manifest_v2.csv')
setup.save_json(RUN_DIR / 'input_fingerprints.json', expected)
trained_folds = {}
validation_rows = []
try:
    for fold in (1, 2, 3):
        print(f'Fold {fold}/3: loading and fitting...', flush=True)
        data = setup.make_fold(train_records, manifest, similarity, fold, multilingual=False)
        train = data['inner_train']; val = data['inner_validation']
        setup.require(all(r['language'] == 'EN' for r in train), 'Non-English training record.')
        fold_dir = RUN_DIR / f'fold_{fold}'; fold_dir.mkdir()
        fit_started = time.perf_counter()
        vectorizer = TfidfVectorizer(**BASELINE_CONFIG['vectorizer'])
        X_train = vectorizer.fit_transform([r['text'] for r in train])
        X_val = vectorizer.transform([r['text'] for r in val])
        models = {}; thresholds = {}; probs = {}; searches = {}
        for level in ('coarse', 'fine'):
            y = setup.binary_labels([r[level] for r in train], taxonomy[level])
            models[level] = fit_binary_models(X_train, y)
        fit_seconds = time.perf_counter() - fit_started
        tuning_started = time.perf_counter()
        for level in ('coarse', 'fine'):
            probs[level] = predict_probabilities(models[level], X_val)
            thresholds[level], searches[level] = choose_threshold([r[level] for r in val],probs[level],taxonomy[level])
            best = max(searches[level],key=lambda r:(r['validation_sample_f1'],r['threshold']))
            validation_rows.append({'fold':fold,'level':level,'threshold':thresholds[level],
                                    'validation_F1':best['validation_sample_f1'],
                                    'train_documents':len(train),'validation_documents':len(val)})
        tuning_seconds = time.perf_counter() - tuning_started
        bundle = {'vectorizer':vectorizer,'models':models,'thresholds':thresholds,
                  'classes':{k:taxonomy[k] for k in ('coarse','fine')},
                  'train_ids':[r['document_id'] for r in train],
                  'validation_ids':[r['document_id'] for r in val]}
        joblib.dump(bundle, fold_dir / 'tfidf_models.joblib')
        setup.save_json(fold_dir / 'threshold_search.json', searches)
        setup.save_json(fold_dir / 'timing.json',{'vectorize_and_fit_seconds':fit_seconds,'validation_predict_and_tune_seconds':tuning_seconds})
        np.savez_compressed(fold_dir / 'validation_probabilities.npz',
                            ids=np.array(bundle['validation_ids']),coarse=probs['coarse'],fine=probs['fine'])
        trained_folds[fold] = fold_dir
        print(f'Fold {fold}: complete; thresholds coarse={thresholds["coarse"]}, fine={thresholds["fine"]}',flush=True)
    pd.DataFrame(validation_rows).to_csv(RUN_DIR / 'validation_selection.csv',index=False)
    setup.save_json(RUN_DIR / 'status.json',{'state':'training_and_selection_complete','outer_evaluated':False})
except Exception as exc:
    setup.save_json(RUN_DIR / 'status.json',{'state':'failed_training','error':repr(exc)})
    raise
display(pd.DataFrame(validation_rows))
print('TRAINING_OK — soglie congelate. Risultati mostrati: validazione interna, NON valutazione finale.')
print('Experiment directory:', RUN_DIR)


## 11. Outer evaluation and aggregation

Evaluation loads the models from this run, verifies partition identifiers and applies the frozen thresholds. The three outer partitions provide one prediction for each of the 399 English articles per system. Scores are computed from concatenated predictions rather than unweighted averages of fold scores.

`F1_fine` is the primary score over the 96-label project taxonomy. `F1_fine_v3` is the separate compatibility score using the original scorer vocabulary. Fine-label macro F1 includes only labels with positive gold support in the pooled outer predictions and reports their number. No thresholds are selected on outer data.

In [ ]:
setup.require(set(trained_folds)=={1,2,3}, 'All three training folds must be complete.')
official_classes = {level:(KIT_DIR/'official_v3'/filename).read_text(encoding='utf-8').splitlines()
                    for level,filename in [('coarse','subtask2_narratives.txt'),('fine','subtask2_subnarratives.txt')]}
systems = ['Other_constant','TFIDF_threshold_0.5','TFIDF_threshold_tuned']
all_predictions = {system:[] for system in systems}
all_outer = []
for fold in (1,2,3):
    data = setup.make_fold(train_records,manifest,similarity,fold,multilingual=False)
    outer = data['outer_evaluation']; fold_dir=trained_folds[fold]
    # Load only models created by this run; never load untrusted joblib files.
    bundle = joblib.load(fold_dir/'tfidf_models.joblib')
    setup.require(all(bundle['classes'][k]==taxonomy[k] for k in ('coarse','fine')), 'Class order changed.')
    setup.require(bundle['train_ids'] == [r['document_id'] for r in data['inner_train']], 'Training identifiers changed.')
    setup.require(bundle['validation_ids'] == [r['document_id'] for r in data['inner_validation']], 'Validation identifiers changed.')
    held_ids={r['document_id'] for r in outer}
    setup.require(not held_ids.intersection(bundle['train_ids']+bundle['validation_ids']), 'Training/validation identifiers overlap outer evaluation.')
    started=time.perf_counter()
    X_outer=bundle['vectorizer'].transform([r['text'] for r in outer])
    outer_probs={k:predict_probabilities(bundle['models'][k],X_outer) for k in ('coarse','fine')}
    elapsed=time.perf_counter()-started
    setup.save_json(fold_dir/'outer_inference_timing.json',{'vectorize_and_predict_seconds':elapsed,'documents':len(outer)})
    np.savez_compressed(fold_dir/'outer_probabilities.npz',ids=np.array([r['document_id'] for r in outer]),**outer_probs)
    all_outer.extend(outer)
    for system in systems:
        predicted={}
        for level in ('coarse','fine'):
            if system=='Other_constant':predicted[level]=[['Other'] for _ in outer]
            else:
                threshold=.5 if system=='TFIDF_threshold_0.5' else bundle['thresholds'][level]
                predicted[level]=decode_probabilities(outer_probs[level],threshold,taxonomy[level])
        write_predictions(fold_dir/(system+'.tsv'),outer,predicted['coarse'],predicted['fine'])
        all_predictions[system].extend([{'id':r['document_id'],'fold':fold,'coarse':c,'fine':f}
            for r,c,f in zip(outer,predicted['coarse'],predicted['fine'])])
setup.require(len(all_outer)==399 and len({r['document_id'] for r in all_outer})==399,'Incomplete outer coverage.')
all_outer=sorted(all_outer,key=lambda r:r['document_id'])
metric_rows=[]
per_document=[]
for system in systems:
    ordered=sorted(all_predictions[system],key=lambda r:r['id'])
    setup.require([r['id'] for r in ordered]==[r['document_id'] for r in all_outer], 'Prediction identifiers are not aligned.')
    predicted={level:[r[level] for r in ordered] for level in ('coarse','fine')}
    metric_rows.append({'system':system,'n_documents':399,**evaluation_metrics(all_outer,predicted,official_classes)})
    write_predictions(RUN_DIR/(system+'_out_of_fold.tsv'),all_outer,predicted['coarse'],predicted['fine'])
    for gold,pred in zip(all_outer,ordered):
        row={'system':system,'id':gold['document_id'],'fold':pred['fold']}
        for level in ('coarse','fine'):
            row['F1_'+level]=setup.sample_f1_strict([gold[level]],[pred[level]],taxonomy[level])['mean']
        per_document.append(row)
metrics_table=pd.DataFrame(metric_rows)
metrics_table.to_csv(RUN_DIR/'metrics.csv',index=False)
pd.DataFrame(per_document).to_csv(RUN_DIR/'per_document_scores.csv',index=False)
setup.save_json(RUN_DIR/'metrics.json',metric_rows)
setup.save_json(RUN_DIR/'status.json',{'state':'complete','outer_evaluated':True,'course_test_evaluated':False})
display(metrics_table[['system','n_documents','F1_coarse','F1_fine','F1_fine_v3','micro_F1_fine','macro_F1_fine_gold_supported']])
print('VALUTAZIONE_OK — 399 predizioni esterne per sistema; nessun tuning sui punteggi esterni.')


## 12. Export results

The summary records configuration, environment, validation selections and pooled outer metrics. A ZIP archive contains the complete experiment directory, including fitted models and predictions.

This experiment uses one seed and does not estimate training variability or confidence intervals. Subsequent comparisons that adapt to inspected outer results must be reported as exploratory.

In [ ]:
baseline_summary = {
    'status':'BASELINES_OK','split_version':'v2','taxonomy_version':'project_v1_96fine',
    'training_language':'EN','seed':42,'outer_documents':399,
    'systems':metric_rows,'validation_selection':validation_rows,
    'course_test_labels_loaded':False,'course_test_evaluated':False,
    'configuration':BASELINE_CONFIG,'environment':environment,
    'notes':['Both TF-IDF decoders use argmax fallback.',
             'Out-of-fold results use project taxonomy; v3 compatibility scores are separate.',
             'No confidence intervals or multi-seed stability claims in this step.']}
setup.save_json(RUN_DIR/'baseline_summary.json',baseline_summary)
zip_path=shutil.make_archive(str(RUN_DIR), 'zip', RUN_DIR)
print(json.dumps(baseline_summary,indent=2,ensure_ascii=False))
print('Summary:',RUN_DIR/'baseline_summary.json')
print('Experiment archive:',zip_path)
